In [36]:
from graph_transformer_long_range_niches.pp.datasets import prepare_geome_dataset
from graph_transformer_long_range_niches.model.baseline import BaselinePCA
from graph_transformer_long_range_niches.tl.load_config import Config

from torch_geometric.data.lightning import LightningDataset
from sklearn.model_selection import train_test_split

import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

# Global functions

In [37]:
def calculate_result_metric(y_train, y_pred_train, y_test, y_pred_test):

    # Calculate result metric
    mae_train = mean_absolute_error(y_train, y_pred_train)
    mae_test = mean_absolute_error(y_test, y_pred_test)
    print(f"Mean Absolute Error: train={np.round(mae_train, 2)}, test={np.round(mae_test, 2)}")

    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)
    print(f"R-squared Score: train={np.round(r2_train, 2)}, test={np.round(r2_test, 2)}")
    
    residuals_train = y_train - y_pred_train
    residuals_test = y_test - y_pred_test
    print(f"Standard Deviation of Residuals: train={np.round(np.std(residuals_train), 2)}, test={np.round(np.std(residuals_test), 2)}")

# Load data

In [38]:
cfg_path_ct_lung5 = '/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src/config_files/he23_gnn_ct_lung5.yaml'
cfg_path_niche_lung5 = '/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src/config_files/he23_gnn_niche_lung5.yaml'

In [39]:
cfg = Config(cfg_path_ct_lung5)

Load cfg file...
{'out_dir': 'results', 'wandb': {'use': True, 'project_name': 'GTLongRange', 'name': 'he23_ct_lung5_gnn', 'run_idx': None}, 'model': {'model_type': 'gnn', 'n_epochs': 100}, 'optim': {'lr': 0.001, 'wd': '1e-3', 'warm_up': 10}, 'dataset': {'h5ad_data': '/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/unprocessed_data/he22_cosmx_human_lung.h5ad', 'prediction_task': 'node', 'prediction_obs': 'author_cell_type', 'library_key': 'window', 'subset_dict': {'donor_id': ['Lung5']}, 'spatial_neigbors_kwargs': {'radius': 30, 'coord_type': 'generic'}, 'batch_size': 10, 'train_size': 0.8, 'val_size': 0.2, 'test_size': 0.0}, 'gnn': {'gnn_type': 'GCN', 'num_layers': 2, 'hidden_dim': 256, 'embed_dim': 256, 'dropout': 0.1}}


In [40]:
pyg_datas = prepare_geome_dataset(cfg)
train_size, val_size, test_size = float(cfg.get('dataset/train_size')), float(cfg.get('dataset/val_size')), float(cfg.get('dataset/test_size'))
train_ds, val_ds = train_test_split(pyg_datas, train_size=train_size, test_size=val_size+test_size, random_state=42)
print(f'train ds: {len(train_ds)}, val ds: {len(val_ds)}')

/home/icb/francesca.drummer/1-Projects/geome/src/geome/transforms/categorize.py:36: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  getattr(adata, self.axis)[key] = getattr(adata, self.axis)[key].astype("category")


[Data(x=[1169, 960], edge_index=[2, 42], y=[1169, 18]), Data(x=[2573, 960], edge_index=[2, 244], y=[2573, 18]), Data(x=[2527, 960], edge_index=[2, 206], y=[2527, 18])]
141
train ds: 112, val ds: 29


In [41]:
def create_x_y(datas):
    X_train = []  # List to store features
    y_train = []  # List to store labels
    for data in datas:
        # Extract features
        x = data.x.numpy()  # Assuming your features are stored in .x attribute
        X_train.append(x)
        
        # Extract labels
        y = data.y.numpy()  # Assuming your labels are stored in .y attribute
        y_train.append(y)

    # Convert lists to numpy arrays
    X_train = np.vstack(X_train)
    y_train = np.concatenate(y_train)
    return X_train, y_train

In [42]:
X_train, y_train = create_x_y(train_ds)
print(X_train.shape, y_train.shape)

(245950, 960) (245950, 18)


In [43]:
X_test, y_test = create_x_y(val_ds)
print(X_test.shape, y_test.shape)

(55651, 960) (55651, 18)


# Baselines

## Random

In [44]:
num_classes = len(y_test[1])
y_pred_train = np.random.randint(0, num_classes, len(y_train))
y_pred_test= np.random.randint(0, num_classes, len(y_test))

In [45]:
train_acc = accuracy_score(np.argmax(y_train, axis=1), y_pred_train)
test_acc = accuracy_score(np.argmax(y_test, axis=1), y_pred_test)
print(f'Train acc: {np.round(train_acc, 2)}, Test acc: {np.round(test_acc, 2)}')

Train acc: 0.06, Test acc: 0.06


**RESULTS**

- Lung5 - cell type: Train acc: 0.06, Test acc: 0.06
- Lung5 - niche: Train acc: 0.11, Test acc: 0.11


## PCA

In [46]:
baseline_pca = BaselinePCA()

In [47]:
y_pred_train, y_pred_test = baseline_pca.run(X_train, y_train, X_test, y_test)

In [48]:
np.argmax(y_pred_train, axis=1)

array([ 8, 17, 11, ..., 14, 14, 14])

In [49]:
train_acc = accuracy_score(np.argmax(y_train, axis=1), np.argmax(y_pred_train, axis=1))
test_acc = accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_pred_test, axis=1))
print(f'Train acc: {np.round(train_acc, 2)}, Test acc: {np.round(test_acc, 2)}')

Train acc: 0.78, Test acc: 0.76


**RESULTS**

- Lung5 - cell type: Train acc: 0.78, Test acc: 0.76
- Lung5 - niche: Train acc: 0.57, Test acc: 0.47

## FCNN

Run as script.

Results: 

- he23_fcnn_**ct**_lung5:
   - val_acc = 0.93     
   - val_f1 = 0.93
   - val_loss = 0.2
- he23_fcnn_**niche**_lung5
   - val_acc = 0.61     
   - val_f1 = 0.61
   - val_loss = 1.11